In [0]:
# Import required libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np

In [0]:
# Get Snowflake password from Databricks Secrets
sfPassword = dbutils.secrets.get(scope="snowflake-creds", key="snowflake-password")

In [0]:
# Read data from Snowflake
df = spark.read \
  .format("snowflake") \
  .options(
      sfUrl="https://kyihiun-lp37397.snowflakecomputing.com",
      sfUser="HARRY134",
      sfPassword=sfPassword,
      sfDatabase="MLPIPELINE",
      sfSchema="DBT_HJONES",
      sfWarehouse="COMPUTE_WH"
  ) \
  .option("dbtable", "FCT_FEATURES") \
  .load()

# Display first few rows to verify
display(df.limit(5))

DURATION,SRC_BYTES,DST_BYTES,IS_TCP,IS_UDP,IS_HTTP,IS_SMTP,IS_FLAG_SF,IS_ATTACK
0,491,0,1,0,0,0,1,0
0,146,0,0,1,0,0,1,0
0,0,0,1,0,0,0,0,1
0,232,8153,1,0,1,0,1,0
0,199,420,1,0,1,0,1,0


In [0]:
# Convert to Pandas
pandas_df = df.toPandas()

# Convert IS_ATTACK to integer
pandas_df['IS_ATTACK'] = pandas_df['IS_ATTACK'].astype(int)

# Separate features and target
X = pandas_df.drop('IS_ATTACK', axis=1)
y = pandas_df['IS_ATTACK']

print(f"Features: {X.shape[1]} columns")
print(f"Samples: {X.shape[0]} rows")

Features: 8 columns
Samples: 101158 rows


In [0]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predict and evaluate
y_pred = rf_model.predict(X_test)

print("MODEL PERFORMANCE")
print("-" * 40)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

MODEL PERFORMANCE
----------------------------------------
Accuracy:  0.9843

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     12992
           1       0.99      0.96      0.98      7240

    accuracy                           0.98     20232
   macro avg       0.99      0.98      0.98     20232
weighted avg       0.98      0.98      0.98     20232

